# P01 — Introduction to Computer Vision
**Course:** Advanced Computer Vision — Dr. Arya Adhyaksa Waskita  
**Session:** P01 — Introduction to Computer Vision  
**Assignment:** Pre-Lecture for P02 (p. 18/20)

Notebook ini memenuhi 4 tugas pra-perkuliahan dan dapat dijalankan langsung di **Kaggle Notebook** (Code) tanpa instalasi tambahan. Kaggle sudah menyediakan PyTorch, OpenCV, dan Jupyter.

**Cara pakai di Kaggle:**
1. Upload file `.ipynb` ini sebagai Kaggle Notebook atau copy-paste per cell
2. Run All — semua cell akan mencetak versi, CUDA, dan demo P02 (filter, edge, warna)

In [ ]:
# Cell 1 — Environment Check (Hal. 15)
# Tugas 1 & 3: Setup environment + testing script
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.cuda")

import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Deteksi GPU yang kompatibel (P100 sm_60 tidak support PyTorch 2.x sm_70+)
def get_compatible_device():
    if torch.cuda.is_available():
        try:
            # Coba alokasi kecil untuk cek kompatibilitas
            torch.zeros(1, device="cuda")
            return "cuda"
        except Exception as e:
            print(f"CUDA warning: {e}")
            print("Fallback to CPU — P100 (sm_60) tidak kompatibel dengan PyTorch ini (butuh sm_70+)")
            return "cpu"
    return "cpu"

DEVICE = get_compatible_device()

print("=== Environment Check ===")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    if torch.cuda.is_available():
        print(f"GPU detected but incompatible (P100 sm_60) — using CPU")
    else:
        print("CUDA not available — using CPU")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")
print("Matplotlib version:", plt.matplotlib.__version__)


In [ ]:
# Cell 2 — Convolution Test (Hal. 15)
# Tugas 3: Cek Conv output shape — pakai DEVICE yang kompatibel
x = torch.randn(2, 3, 224, 224, device=DEVICE)
conv = nn.Conv2d(3, 64, kernel_size=3, padding=1).to(DEVICE)
y = conv(x)
print(f"Input shape: {x.shape} on {DEVICE}")
print(f"Conv output shape: {y.shape}")
assert y.shape == torch.Size([2, 64, 224, 224]), "Shape mismatch!"
print(f"Conv test PASSED on {DEVICE} — environment siap untuk P02 (Spatial Filtering & Convolution)")


In [ ]:
# Cell 3 — Preview P02: Spatial Filtering, Edge Detection, Color Space
# Tugas 2 & P02: Fondasi Pemrosesan Citra Digital
# Membuat citra dummy agar bisa jalan tanpa dataset eksternal (Kaggle-friendly)

# Buat citra dummy 256x256 dengan gradien + kotak
img = np.zeros((256, 256, 3), dtype=np.uint8)
for i in range(256):
    img[:, i] = int(i * 0.8)  # gradien horizontal
cv2.rectangle(img, (60, 60), (180, 180), (255, 255, 255), -1)
cv2.circle(img, (128, 128), 40, (0, 0, 255), -1)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 1. Spatial Filtering — Gaussian Blur (Convolution)
blur = cv2.GaussianBlur(gray, (9, 9), 0)

# 2. Edge Detection — Sobel & Canny
sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel = np.sqrt(sobelx**2 + sobely**2)
sobel = np.uint8(sobel / sobel.max() * 255)
canny = cv2.Canny(gray, 100, 200)

# 3. Transformasi Ruang Warna — RGB, HSV, Lab
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)

print(f"Gray shape: {gray.shape}, Blur shape: {blur.shape}")
print(f"Sobel shape: {sobel.shape}, Canny shape: {canny.shape}")
print(f"HSV shape: {hsv.shape}, Lab shape: {lab.shape}")

# Visualisasi
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0,0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0,0].set_title("Original (BGR->RGB)"); axes[0,0].axis("off")
axes[0,1].imshow(gray, cmap="gray"); axes[0,1].set_title("Grayscale"); axes[0,1].axis("off")
axes[0,2].imshow(blur, cmap="gray"); axes[0,2].set_title("Gaussian Blur 9x9"); axes[0,2].axis("off")
axes[0,3].imshow(sobel, cmap="gray"); axes[0,3].set_title("Sobel Edge"); axes[0,3].axis("off")
axes[1,0].imshow(canny, cmap="gray"); axes[1,0].set_title("Canny Edge"); axes[1,0].axis("off")
axes[1,1].imshow(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)); axes[1,1].set_title("HSV"); axes[1,1].axis("off")
axes[1,2].imshow(lab); axes[1,2].set_title("Lab"); axes[1,2].axis("off")
axes[1,3].axis("off"); axes[1,3].text(0.5, 0.5, "P02 Preview\nSelesai", ha="center", va="center", fontsize=12)
plt.tight_layout()
plt.show()
print("Preview P02 PASSED — Spatial Filtering, Edge Detection, Color Space berhasil")

In [ ]:
# Cell 4 — Jupyter Check (Tugas 4)
# Tugas 4: Install dan familiarisasi Jupyter Notebook
import jupyter_core
import notebook
print(f"Jupyter Core: {jupyter_core.__version__}")
try:
    print(f"Notebook: {notebook.__version__}")
except:
    print("Notebook version: available via jupyter lab")
print("Jupyter Notebook siap — di Kaggle, Jupyter sudah aktif otomatis")
print("Tips: Shift+Enter untuk run cell, Esc+M untuk markdown, Esc+Y untuk code")

In [ ]:
# Cell 5 — Ringkasan Szeliski Bab 2-3 (Tugas 2)
# Tugas 2: Baca Bab 2-3 Szeliski "Computer Vision: Algorithms and Applications"
# Ringkasan singkat untuk laporan

print("""
=== Ringkasan Szeliski Bab 2-3 ===

Bab 2 — Image Formation:
- Model kamera pinhole, lensa, dan proyeksi perspektif
- Radiometri: interaksi cahaya-materi, BRDF, shading
- Sampling & aliasing, color filter array (Bayer)

Bab 3 — Image Processing:
- Point operators: brightness/contrast, histogram equalization, gamma
- Linear filtering: convolution, Gaussian, Sobel, Laplacian
- Non-linear: median filter, bilateral filter
- Fourier transform, pyramids (Gaussian/Laplacian), warping

Relevansi P02: Bab 3 adalah fondasi untuk Spatial Filtering, Convolution,
Edge Detection (Canny/Sobel), dan Color Space yang akan dipraktikkan di P02.
Sumber: Szeliski, Computer Vision: Algorithms and Applications, 2nd Ed.
""")

## Checklist Tugas P01

- [x] **Tugas 1:** Setup environment — Python 3.10, PyTorch, OpenCV, Jupyter (cek Cell 1)
- [x] **Tugas 2:** Baca Szeliski Bab 2-3 — ringkasan di Cell 5
- [x] **Tugas 3:** Testing environment — `torch`, `cv2`, `Conv output shape` (Cell 1-2)
- [x] **Tugas 4:** Jupyter Notebook — verifikasi di Cell 4

**Output yang diharapkan dosen:**
- `PyTorch version: 2.x.x`, `CUDA available: True/False`, `OpenCV version: 4.x.x`, `Conv output shape: torch.Size([2, 64, 224, 224])`
- Screenshot atau export PDF notebook sebagai bukti

**Catatan Kaggle:**
- Jika `CUDA available: False`, aktifkan GPU di Kaggle: Settings > Accelerator > GPU T4 x2
- Tidak perlu `pip install` — semua library sudah tersedia di Kaggle image
- Untuk Colab, tambahkan `!pip install opencv-python-headless` jika perlu